<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Build step 3</div>
<div style="font-size:24px;font-weight:700;">&#129351; Gold: business-ready tables</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Store, product, hourly, inventory and customer tables at the grain each business question needs, plus the promo target list.</p>


In [0]:
# Parameters come from the bundle's build job: the catalog and the three schemas the bundle created.
# This sets USE CATALOG and the session variables bronze_schema, silver_schema and gold_schema.
from bean_and_brew import init, fill

cfg = init(spark, dbutils)


In [0]:
%sql
-- Gold: one table per business grain
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.store_daily_performance')
COMMENT 'Gold: daily store P&L-lite. One row per store per business date with traffic, sales, margin, loyalty mix, and labor. Excludes voided tickets. Primary table for store performance dashboards, Genie, and alerts. Sources: silver.fact_transactions, silver.fact_labor_daily, silver.dim_store.'
AS
WITH t AS (
  SELECT store_id, business_date,
    count(*) transactions, sum(item_qty) items_sold, sum(gross_sales) gross_sales, sum(discount_amount) discount_amount,
    sum(net_sales) net_sales, sum(cost_of_goods) cost_of_goods,
    count_if(is_loyalty_transaction) loyalty_transactions,
    count_if(order_channel IN ('Mobile App','Delivery')) digital_transactions,
    count_if(order_channel = 'Drive-Thru') drive_thru_transactions
  FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided GROUP BY ALL)
SELECT t.business_date, t.store_id, s.store_name, s.city, s.state, s.region, s.store_format,
  datediff(t.business_date, s.open_date) >= 365 AS is_comp_store,
  t.transactions, t.items_sold, t.gross_sales, t.discount_amount, t.net_sales, t.cost_of_goods,
  t.net_sales - t.cost_of_goods AS gross_profit,
  cast(round(t.net_sales / t.transactions, 2) AS DECIMAL(10,2)) AS avg_ticket,
  t.loyalty_transactions, t.digital_transactions, t.drive_thru_transactions,
  l.worked_hours AS labor_hours, l.labor_cost,
  cast(round(l.labor_cost / t.net_sales, 4) AS DECIMAL(8,4)) AS labor_cost_pct
FROM t JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id)
LEFT JOIN IDENTIFIER(silver_schema || '.fact_labor_daily') l ON l.store_id = t.store_id AND l.business_date = t.business_date;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.product_sales_daily')
COMMENT 'Gold: daily product sales by store. One row per business date, store, and product with units, sales, and margin. Sources: silver.fact_sales_line_items, silver.dim_product, silver.dim_store.'
AS SELECT f.business_date, f.store_id, s.store_name, s.region, f.product_id, p.product_name, p.category, p.serving_type, p.is_beverage,
  sum(f.quantity) units_sold, sum(f.gross_sales) gross_sales, sum(f.discount_amount) discount_amount, sum(f.net_sales) net_sales,
  sum(f.cost_of_goods) cost_of_goods, sum(f.net_sales) - sum(f.cost_of_goods) gross_profit
FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') f JOIN IDENTIFIER(silver_schema || '.dim_product') p USING (product_id) JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id)
GROUP BY ALL;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.store_hourly_traffic')
COMMENT 'Gold: hourly ticket counts and sales per store and day, for staffing and peak-hour analysis. Excludes voided tickets. Source: silver.fact_transactions.'
AS SELECT store_id, business_date, business_hour, date_format(business_date,'EEEE') AS day_of_week,
  count(*) transactions, sum(net_sales) net_sales, count_if(order_channel = 'Mobile App') mobile_transactions
FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided GROUP BY ALL;

CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.inventory_current_status')
COMMENT 'Gold: latest inventory position per store and stock item with stock health status (OK / LOW / REORDER NOW). Source: silver.fact_inventory_daily, silver.dim_store.'
AS
WITH latest AS (SELECT * FROM IDENTIFIER(silver_schema || '.fact_inventory_daily') QUALIFY row_number() OVER (PARTITION BY store_id, item_code ORDER BY count_date DESC) = 1)
SELECT l.store_id, s.store_name, s.region, l.count_date AS as_of_date, l.item_code, l.item_name, l.unit_of_measure,
  l.on_hand_qty, l.par_level, l.reorder_point, cast(round(l.on_hand_qty / l.par_level, 3) AS DECIMAL(6,3)) AS pct_of_par,
  l.inventory_value,
  CASE WHEN l.on_hand_qty < l.reorder_point THEN 'REORDER NOW' WHEN l.on_hand_qty < l.reorder_point * 1.5 THEN 'LOW' ELSE 'OK' END AS stock_status
FROM latest l JOIN IDENTIFIER(silver_schema || '.dim_store') s USING (store_id);

-- mode(x, true) breaks ties on the lowest value, so favorite store/product are stable across runs
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.customer_360')
COMMENT 'Gold: one row per Bean & Brew Rewards member with lifetime value, visit frequency, recency, favorite store/product, and value segment (Champion = top 10% by lifetime spend, Loyal Regular = next 30%, Occasional = next 40%, Light = bottom 20%). Sources: silver.dim_loyalty_member, silver.fact_transactions, silver.fact_sales_line_items, silver.dim_product.'
AS
WITH tx AS (
  SELECT loyalty_member_id member_id, count(*) visits, sum(net_sales) lifetime_net_sales, min(business_date) first_visit_date, max(business_date) last_visit_date,
    mode(store_id, true) favorite_store_id, count_if(order_channel = 'Mobile App') mobile_visits
  FROM IDENTIFIER(silver_schema || '.fact_transactions') WHERE NOT is_voided AND loyalty_member_id IS NOT NULL GROUP BY 1),
fav AS (
  SELECT loyalty_member_id member_id, mode(p.product_name, true) favorite_product
  FROM IDENTIFIER(silver_schema || '.fact_sales_line_items') f JOIN IDENTIFIER(silver_schema || '.dim_product') p USING (product_id) WHERE loyalty_member_id IS NOT NULL GROUP BY 1),
base AS (
  SELECT m.member_id, m.first_name, m.last_name, m.loyalty_tier, m.signup_date, m.home_store_id,
    tx.visits, tx.lifetime_net_sales, cast(round(tx.lifetime_net_sales / tx.visits, 2) AS DECIMAL(10,2)) avg_ticket,
    tx.first_visit_date, tx.last_visit_date, datediff(DATE'2026-10-19', tx.last_visit_date) AS days_since_last_visit,
    tx.favorite_store_id, fav.favorite_product, round(tx.mobile_visits / tx.visits, 3) AS mobile_share,
    percent_rank() OVER (ORDER BY tx.lifetime_net_sales DESC, m.member_id) pr
  FROM IDENTIFIER(silver_schema || '.dim_loyalty_member') m LEFT JOIN tx USING (member_id) LEFT JOIN fav USING (member_id))
SELECT * EXCEPT (pr),
  CASE WHEN pr < 0.10 THEN 'Champion' WHEN pr < 0.40 THEN 'Loyal Regular' WHEN pr < 0.80 THEN 'Occasional' ELSE 'Light' END AS customer_segment
FROM base;

In [0]:
%sql
-- Same logic as the "Loyalty Promo Target List Prep" visual data prep flow (deployed below), so the
-- table exists before the demo. Running the flow live overwrites it with identical rows.
-- CHAR(36) is '$': a literal dollar-digit is read as a parameter in notebook SQL.
CREATE TABLE IF NOT EXISTS IDENTIFIER(gold_schema || '.loyalty_promo_targets') AS
SELECT c.member_id, c.first_name, c.loyalty_tier, c.customer_segment, c.lifetime_net_sales, c.visits, c.favorite_product,
       s.store_name, s.region,
       CASE WHEN c.favorite_product LIKE '%Cold Brew%' THEN 'Free Nitro Upgrade'
            WHEN c.favorite_product LIKE '%Latte%'     THEN 'Free Oat Milk Swap'
            ELSE CONCAT(CHAR(36), '2 Off Pastry') END AS promo_offer
FROM IDENTIFIER(gold_schema || '.customer_360') c
JOIN IDENTIFIER(silver_schema || '.dim_store') s ON c.home_store_id = s.store_id
WHERE c.customer_segment IN ('Champion', 'Loyal Regular') AND c.loyalty_tier IN ('Gold', 'Reserve');